In [1]:
import sqlite3, pandas as pd
con = sqlite3.connect("practice.db")
con.executescript(open("practice_setup.sql").read())
def run(q): return pd.read_sql(q, con)

In [2]:
"""-- 1. Filter + aggregate + HAVING  [Service / aggregation]
-- Return dept_id and total salary per department,
-- only for departments whose total salary is above 150000,
-- sorted by total salary descending.
"""
query1 = """SELECT dept_id,SUM(salary) as 'total' from employees group by dept_id having total>150000 order by total desc"""
run(query1)

,dept_id,total
0,1,386000
1,2,265000


In [7]:
"""-- 2. 3-table join + aggregate  [AI screen / joins]
-- Return each project_name and the total hours worked on it
-- (sum across all employees assigned), highest total first.
-- Projects with no one assigned can be left out.
"""
query2 = """SELECT p.project_name,SUM(a.hours) as 'total_hours' from assignments a JOIN projects p on p.project_id = a.project_id
join employees e on e.emp_id=a.emp_id
group by p.project_name
order by total_hours desc"""
run(query2)

,project_name,total_hours
0,Payment Gateway,480
1,Mobile App,450
2,CRM Rollout,370
3,Audit System,280
4,Q4 Campaign,180


In [16]:
"""-- 3. Correlated subquery  [AI screen / subquery]
-- Return the name and salary of every employee who earns the
-- HIGHEST salary within their own department.
-- (Think: salary = the max salary of employees in the same dept.)"""
query3 = """SELECT name,salary from employees e1 where salary in (select MAX(salary) from employees e2 where e1.dept_id = e2.dept_id)"""
run(query3)

,name,salary
0,Arjun,95000
1,Sneha,88000
2,Anita,50000
3,Kavya,76000
4,Deepa,59000
5,Nisha,52000


In [10]:
"""-- 4. LAG window  [AI screen / window]
-- For emp_id 5's sales, return sale_date, amount, and the PREVIOUS
-- sale's amount in a column "prev_amount", ordered by sale_date.
-- The first row's prev_amount will be NULL."""
query4 = """ SELECT sale_date, amount, LAG(amount) over(order by sale_date ASC) as 'prev_amount' from sales"""
run(query4)

,sale_date,amount,prev_amount
0,2024-01-05,12000,NaN
1,2024-01-12,25000,12000.0
2,2024-01-20,8000,25000.0
3,2024-02-03,15000,8000.0
4,2024-02-15,30000,15000.0
5,2024-02-28,11000,30000.0
6,2024-03-10,18000,11000.0
7,2024-03-18,22000,18000.0
8,2024-03-25,9000,22000.0
9,2024-04-02,14000,9000.0


In [15]:
"""-- 5. Join + group + count + HAVING  [AI screen / aggregation]
-- Return dept_name and the number of employees in each department,
-- only for departments with MORE THAN 2 employees,
-- sorted by that count descending. Use the department NAME."""
query5 = """SELECT d.dept_name,count(e.name) as 'cnt' from departments d JOIN employees e on d.dept_id=e.dept_id group by d.dept_name having cnt > 2
order by cnt desc"""
run(query5)

,dept_name,cnt
0,Engineering,5
1,Sales,4


In [17]:
"""-- 1. [AI screen / correlated subquery]
-- Return name, dept_id, salary of every employee who earns MORE than
-- the average salary OF THEIR OWN department.
-- (subquery must reference the outer row's dept)"""
query5 = """SELECT NAME,dept_id,salary from employees e1 where salary > (select AVG(salary) from employees e2 where e1.dept_id = e2.dept_id)"""
run(query5)

,name,dept_id,salary
0,Arjun,1,95000
1,Divya,1,81000
2,Sneha,2,88000
3,Anita,3,50000
4,Kavya,4,76000


In [18]:
"""-- 2. [AI screen / correlated subquery]
-- Return name and hire_date of the EARLIEST hire in each department.
-- (hire_date is the MIN hire_date among employees in the same dept)
"""
query6 = """SELECT NAME,hire_date from employees e1 where hire_date in (select MIN(hire_date) from employees e2 where e1.dept_id = e2.dept_id)"""
run(query6)

,name,hire_date
0,Arjun,2019-03-15
1,Sneha,2018-09-10
2,Anita,2021-08-19
3,Kavya,2019-12-01
4,Deepa,2022-09-15
5,Nisha,2023-01-10


In [20]:
"""-- 3. [AI screen / correlated subquery]
-- Return each department's name and how many employees it has,
-- computed with a correlated subquery in the SELECT list
-- (SELECT COUNT(*) FROM employees ... matching this dept), not a JOIN.
-- Only departments in the departments table; a 0 is fine if it happens."""
query7 = """SELECT d.dept_name,(select count(e.name) from employees e where e.dept_id=d.dept_id) as "Count" from departments d"""
run(query7)

,dept_name,Count
0,Engineering,5
1,Sales,4
2,HR,2
3,Finance,2
4,Marketing,1


In [22]:
"""-- 4. [AI screen / anti-join]
-- Return the name of every employee who has NO rows in the assignments
-- table (i.e. is assigned to no project). Write it with NOT EXISTS.
"""
query8 = """Select name from employees e1 where NOT EXISTS (select 1 from assignments a where a.emp_id=e1.emp_id)"""
run(query8)

,name
0,Sneha
1,Anita
2,Manoj
3,Nisha


In [23]:
"""-- 5. [AI screen / anti-join]
-- Same question as #4 (employees assigned to nothing),
-- but written the OTHER way: LEFT JOIN assignments, then WHERE ... IS NULL.
-- Return name only."""
query9 = """ Select Name from employees e LEFT JOIN assignments a on a.emp_id = e.emp_id where a.emp_id IS NULL"""
run(query9)

,name
0,Sneha
1,Anita
2,Manoj
3,Nisha


In [34]:
"""-- 6. [AI screen / outer join]
-- Return every project_name and the number of DISTINCT employees assigned
-- to it. Projects with nobody assigned must appear with 0.
-- (which table is the LEFT table? and COUNT what?)"""
query10 = """SELECT p.project_name,count(a.emp_id) as 'cnt' from projects p LEFT JOIN assignments a on a.project_id = p.project_id
group by p.project_name order by cnt desc"""
run(query10)

,project_name,cnt
0,Payment Gateway,3
1,Mobile App,3
2,CRM Rollout,3
3,Audit System,2
4,Q4 Campaign,1
5,Internal Portal,0


In [41]:
"""-- 7. [AI screen / window]
-- Return the TOP 2 earners in EACH department: name, dept_id, salary.
-- (RANK or ROW_NUMBER in a subquery, then filter <= 2.)"""
query11 = """ Select * from (SELECT name,dept_id,salary,RANK() over(partition by dept_id order by salary desc) as 'rank' from employees) as ranked where ranked.rank <=2"""
run(query11)

,name,dept_id,salary,rank
0,Arjun,1,95000,1
1,Divya,1,81000,2
2,Sneha,2,88000,1
3,Lakshmi,2,62000,2
4,Anita,3,50000,1
5,Manoj,3,47000,2
6,Kavya,4,76000,1
7,Suresh,4,64000,2
8,Deepa,5,59000,1
9,Nisha,6,52000,1


In [47]:
"""-- 8. [AI screen / window + join]
-- For the Sales department only (dept_id 2), return each salesperson's
-- name and their TOTAL sales amount, highest first.
-- (join employees to sales, sum per person; a salesperson with no sales
--  can be left out.)"""
query12 = """SELECT e.name,total(s.amount) as 'total_sales' from employees e JOIN sales s on s.emp_id = e.emp_id 
where e.dept_id=2
group by e.name
order by total_sales DESC
"""
run(query12)

,name,total_sales
0,Sneha,77000.0
1,Rohit,65000.0
2,Vikram,33000.0
3,Lakshmi,25000.0
